# Circuit-side runs (CX-QuAPI)

Everything that builds a circuit: how the influence functional is cut, what the cut costs
in gates and in shots, and what the circuit returns. The classical path filter is the other
side of the study and lives in `OFPF.ipynb`.

The `psd` cells are the exception to "any environment": the semidefinite condition is the
only measurement in this tree that needs **cvxpy**. Start the kernel from `~/.venvs/cvx`
for those three cells, or skip them.

Each cell is one step of one system and writes its own files under `Data/<System>/<Property>/`,
exactly as `Running/CXQUAPI/<System>.py` does from the command line. The equivalent command is printed above each
section. Cells carry no dependency on each other with one exception, noted where it applies, so
any single cell may be run on its own once **Master** has been run.

Every cell sets each module global it reads, including the ones another cell changes. A `.py`
driver never had to: it ran in a process of its own, so every module began at its default.


## Master

In [ ]:
import sys

master_dir   = '/home/bjbowling123/Project/CX_QuAPI'
sys.path[:0] = ['/home/bjbowling123/Project/PyQAKE/Code/Package', master_dir + '/Code']

# Seventeen measurement modules read sys.argv for an optional top N. A notebook's argv carries the
# kernel's connection file instead, so int(sys.argv[1]) would raise: it is emptied here and every
# module takes the default written in its own file.
sys.argv     = ['CXQUAPI.ipynb']

from lib.cx_systems_lib import tls, sim2001, chain, ohmic
from measure.CX_coupling_sweep import LAT_TLS, LAT_3LS

# The published systems. TLS_STRONG is the weakly damped system's Hamiltonian on the bath that makes
# the rescaling bite (xi = 1.2, w_c = 2.5, beta = 0.2); CHAIN8 carries the weak system's bath,
# coupling and time step, so that d is the only difference between them.
TLS          = tls()
TLS_STRONG   = dict(tls(), name='TLS-strong', sd=ohmic(1.2, 2.5), beta=0.2)
SIM2001      = sim2001(0.72)
CHAIN8       = chain(8)

# Data/ and Output/ are resolved from lib/paths.py, not from the working directory, so a notebook
# writes exactly where the drivers write. CXQUAPI_ROOT in the environment moves both.
print(master_dir)

## TLS — weakly damped two-level system

`python3 CXQUAPI/TLS.py --skip psd`

In [ ]:
###===================================================<<< Settings

system       = 'TLS'
SY           = TLS
N_dyn        = 5          # one statevector run per N up to here
N_res        = 4          # resource sweep over every group size up to here

###===================================================<<< Calculation

# The audit dump: one run per N, then the resource sweep. N = 6 is a 35-qubit statevector (550 GB)
# and no table reads it, so the sweep stops where the paper stops.
from measure import CX_propagation
CX_propagation.main(system, SY, N_dyn=N_dyn, N_res=N_res)

In [ ]:
###===================================================<<< Settings

system       = 'TLS'
SY           = TLS

###===================================================<<< Calculation

# smallest exact group size g* against N, on the blip lattice of a two-valued s
from measure import CX_gstar_vs_N
CX_gstar_vs_N.SYSTEM = system
CX_gstar_vs_N.CASES  = ((SY, LAT_TLS, 1, 'TLS'),)
CX_gstar_vs_N.main()

In [ ]:
###===================================================<<< Settings

system       = 'TLS'
SY           = TLS
N_max        = 6          # both files carry N = 2..6 in the paper's SI table

###===================================================<<< Calculation

# the Walsh-Fourier dilation against the grouped one, exact and at the optimum
from measure import CX_walsh_vs_lp_exact, CX_walsh_vs_lp_opt
for mod in (CX_walsh_vs_lp_exact, CX_walsh_vs_lp_opt):
    mod.SYSTEM = system
    mod.N_MAX  = N_max
    mod.CASES  = ((SY, 'TLS'),)
    mod.main()

In [ ]:
###===================================================<<< Settings

system       = 'TLS'
SY           = TLS
N_top        = 8          # longest propagation read out

###===================================================<<< Calculation

# full readout against the two truncated ones, with the classical path sum as reference
from measure import CX_results_readout_data
CX_results_readout_data.SYSTEM = system
CX_results_readout_data.CASES  = ((SY, N_top),)
CX_results_readout_data.main()

In [ ]:
###===================================================<<< Settings

# NEEDS cvxpy. The semidefinite condition on the same M, with a bath made Markovian.

system       = 'TLS'
DT, BETA, XI = 0.3, 0.720, 2 * 2.094 / 3.141592653589793
W_C          = 1.0 / (0.25 * DT)       # 0.25 time slices per bath correlation time

###===================================================<<< Calculation

from measure import CX_grouping_psd_vs_lattice as psd_mod
psd_mod.SYSTEM   = system
psd_mod.FILENAME = 'psd_vs_lattice_markov.json'
psd_mod.CASE     = dict(sim2001(BETA), name='Markov-bath on TLS lattice', sd=ohmic(XI, W_C))
psd_mod.LATTICE  = LAT_TLS             # s = (+1, -1): the three-valued lattice, same M
psd_mod.N_Q      = 1                   # one path qubit per slice
psd_mod.N_RANGE  = range(2, 8)         # the module default, restated so the Sim2001 cell cannot shorten it
psd_mod.main()

## TLS-strong — strongly damped two-level system

`python3 CXQUAPI/TLS_strong.py --skip psd`

In [ ]:
###===================================================<<< Settings

system       = 'TLS_strong'
SY           = TLS_STRONG

###===================================================<<< Calculation

# alpha and CNOT of pairs against g-windows for N = 1..7, then g* against N
from measure import CX_gstar_vs_N, CX_grouping_hierarchy
CX_grouping_hierarchy.main()
CX_gstar_vs_N.SYSTEM = system
CX_gstar_vs_N.CASES  = ((SY, LAT_TLS, 1, 'TLS-strong'),)
CX_gstar_vs_N.main()

In [ ]:
###===================================================<<< Settings

system       = 'TLS_strong'
SY           = TLS_STRONG
N_top        = 9          # the program stays affordable to here

###===================================================<<< Calculation

# what a group below g* costs: alpha, its shots, and the gates it saves
from measure import CX_lp_cost_vs_shots, CX_lp_time_vs_shot_time
CX_lp_cost_vs_shots.SYSTEM = system
CX_lp_cost_vs_shots.CASES  = ((SY, LAT_TLS, 1, 'TLS-strong'),)
CX_lp_cost_vs_shots.TOPS   = {'TLS-strong': N_top}
CX_lp_cost_vs_shots.main()
CX_lp_time_vs_shot_time.SYSTEM  = system
CX_lp_time_vs_shot_time.SYSTEMS = {'TLS-strong': SY}
CX_lp_time_vs_shot_time.main()

In [ ]:
###===================================================<<< Settings

system       = 'TLS_strong'
SY           = TLS_STRONG
N_max        = 6          # both files carry N = 2..6 in the paper's SI table

###===================================================<<< Calculation

from measure import CX_walsh_vs_lp_exact, CX_walsh_vs_lp_opt
for mod in (CX_walsh_vs_lp_exact, CX_walsh_vs_lp_opt):
    mod.SYSTEM = system
    mod.N_MAX  = N_max
    mod.CASES  = ((SY, 'TLS-strong'),)
    mod.main()

In [ ]:
###===================================================<<< Settings

system       = 'TLS_strong'
SY           = TLS_STRONG
N_top        = 8
N_rules      = 5          # the propagation length the readout rules are read at

###===================================================<<< Calculation

from measure import CX_results_readout_data, CX_readout_rules
CX_results_readout_data.SYSTEM = system
CX_results_readout_data.CASES  = ((SY, N_top),)
CX_results_readout_data.main()
CX_readout_rules.SYSTEM  = system
CX_readout_rules.SYSTEMS = (SY,)
CX_readout_rules.main(N=N_rules)

In [ ]:
###===================================================<<< Settings

# NEEDS cvxpy. The semidefinite condition against the finite-lattice one, on this system's own bath.

system       = 'TLS_strong'
SY           = TLS_STRONG

###===================================================<<< Calculation

from measure import CX_grouping_psd_vs_lattice as psd_mod
psd_mod.SYSTEM   = system
psd_mod.FILENAME = 'psd_vs_lattice.json'
psd_mod.CASE     = SY
psd_mod.LATTICE  = LAT_TLS
psd_mod.N_Q      = 1
psd_mod.N_RANGE  = range(2, 8)
psd_mod.main()

## Sim2001 — three-state model

`python3 CXQUAPI/Sim2001.py --skip psd`

In [ ]:
###===================================================<<< Settings

system       = 'Sim2001'
SY           = SIM2001

###===================================================<<< Calculation

# g* against N, and what reducing the blip lattice does to it
from measure import CX_gstar_vs_N, CX_lattice_variants
CX_gstar_vs_N.SYSTEM = system
CX_gstar_vs_N.CASES  = ((SY, LAT_3LS, 2, 'Sim2001'),)
CX_gstar_vs_N.main()
CX_lattice_variants.SYSTEM = system
CX_lattice_variants.CASES  = ((SY, 2, 'Sim2001'),)
CX_lattice_variants.main()

In [ ]:
###===================================================<<< Settings

system       = 'Sim2001'
SY           = SIM2001
N_top        = 8

###===================================================<<< Calculation

from measure import CX_lp_cost_vs_shots, CX_lp_time_vs_shot_time
CX_lp_cost_vs_shots.SYSTEM = system
CX_lp_cost_vs_shots.CASES  = ((SY, LAT_3LS, 2, 'Sim2001'),)
CX_lp_cost_vs_shots.TOPS   = {'Sim2001': N_top}
CX_lp_cost_vs_shots.main()
CX_lp_time_vs_shot_time.SYSTEM  = system
CX_lp_time_vs_shot_time.SYSTEMS = {'Sim2001': SY}
CX_lp_time_vs_shot_time.main()

In [ ]:
###===================================================<<< Settings

system       = 'Sim2001'
SY           = SIM2001
# d = 3 at N = 6 is 2e8 CNOT and qiskit's high-level synthesis aborts on it (pyo3 PanicException:
# EdgeIxLimit), which kills the process and the opt pass with it. The scan stops at 5.
N_max        = 5

###===================================================<<< Calculation

from measure import CX_walsh_vs_lp_exact, CX_walsh_vs_lp_opt
for mod in (CX_walsh_vs_lp_exact, CX_walsh_vs_lp_opt):
    mod.SYSTEM = system
    mod.N_MAX  = N_max
    mod.CASES  = ((SY, 'Sim2001'),)
    mod.main()

In [ ]:
###===================================================<<< Settings

system       = 'Sim2001'
SY           = SIM2001
N_top        = 6
N_rules      = 5

###===================================================<<< Calculation

from measure import CX_results_readout_data, CX_readout_rules
CX_results_readout_data.SYSTEM = system
CX_results_readout_data.CASES  = ((SY, N_top),)
CX_results_readout_data.main()
CX_readout_rules.SYSTEM  = system
CX_readout_rules.SYSTEMS = (SY,)
CX_readout_rules.main(N=N_rules)

In [ ]:
###===================================================<<< Settings

system       = 'Sim2001'
SY           = SIM2001
N_top        = 8
ends         = (0, 2)     # |psi> = (|1> + |3>)/sqrt(2), numbered from zero here

###===================================================<<< Calculation

# every population from rho(0) = |psi><psi|, read in full. V = 1 is the model's default now, so this
# is the same system every other cell of this section uses.
from measure import CX_super_pops
CX_super_pops.SYSTEM = system
CX_super_pops.CASES  = ((SY, N_top, ends),)
CX_super_pops.main()

In [ ]:
###===================================================<<< Settings

# NEEDS cvxpy. Two scans: the Markovian bath, then the published one. tab:psd3 is both halves.

system       = 'Sim2001'
SY           = SIM2001
DT, BETA, XI = 0.3, 0.720, 2 * 2.094 / 3.141592653589793
W_C          = 1.0 / (0.25 * DT)       # bath correlation time inside one time step

###===================================================<<< Calculation

from measure import CX_grouping_psd_vs_lattice as psd_mod
psd_mod.LATTICE  = LAT_3LS             # s = (-1, 0, 1), so Delta runs over five values rather than three
psd_mod.N_Q      = 2                   # two path qubits per slice: a group of g costs 2^(4g)
psd_mod.SYSTEM   = system
psd_mod.FILENAME = 'psd_vs_lattice_markov.json'
psd_mod.CASE     = dict(sim2001(BETA), name='Sim2001-markov', sd=ohmic(XI, W_C))
psd_mod.N_RANGE  = range(2, 8)
psd_mod.main()

# The same scan on the published bath. The table ends at N = 6: on this bath no g reaches alpha = 1
# before the Walsh-Fourier one, so N = 7 would only add a 4.3e9-CNOT group no circuit could carry.
psd_mod.FILENAME = 'psd_vs_lattice_ownbath.json'
psd_mod.CASE     = SY
psd_mod.N_RANGE  = range(2, 7)
psd_mod.main()

## Chain8 — eight-level tight-binding chain

`python3 CXQUAPI/Chain8.py`

In [ ]:
###===================================================<<< Settings

system       = 'Chain8'
SY           = CHAIN8
N_top        = 4

###===================================================<<< Calculation

from measure import CX_results_readout_data
CX_results_readout_data.SYSTEM = system
CX_results_readout_data.CASES  = ((SY, N_top),)
CX_results_readout_data.main()

In [ ]:
###===================================================<<< Settings

system       = 'Chain8'
SY           = CHAIN8
N_top        = 5
# |psi> = (|1> + |5>)/sqrt(2). NOT a state symmetric about the centre of the chain: the chain
# reverses and the influence functional is invariant under s -> -s, so such a state would force
# the eight populations into four exactly degenerate pairs.
ends         = (0, 4)

###===================================================<<< Calculation

from measure import CX_super_pops
CX_super_pops.SYSTEM = system
CX_super_pops.CASES  = ((SY, N_top, ends),)
CX_super_pops.main()

## Common — read across the systems

`python3 CXQUAPI/Common.py`

**The only order that matters in this notebook:** `shots` reads the gate counts `cost` writes,
so run the two in that order at least once.

In [ ]:
###===================================================<<< Settings

system       = 'Common'

###===================================================<<< Calculation

# gates and ancillas against the retained memory, for every system at once
from measure import CX_cost_memory_scan
CX_cost_memory_scan.main()

In [ ]:
###===================================================<<< Settings

system       = 'Common'

###===================================================<<< Calculation

# RUN AFTER the cell above: this reads the counts that cost wrote
from measure import CX_shot_cost_data
CX_shot_cost_data.main()

In [ ]:
###===================================================<<< Settings

system       = 'Common'
n_sample     = 0          # exact marginals, which is what Table 3 quotes

###===================================================<<< Calculation

# the readout layer itself at exact marginals, then the fixed-cutoff sweep
from measure import CX_layer_hadamard, CX_fixed_cutoff_sweep
CX_layer_hadamard.SYSTEM   = system
CX_layer_hadamard.N_SAMPLE = n_sample
CX_layer_hadamard.main()
CX_fixed_cutoff_sweep.SYSTEM = system
CX_fixed_cutoff_sweep.main()

In [ ]:
###===================================================<<< Settings

system       = 'Common'

###===================================================<<< Calculation

# one number, the diagonal dominance of M, against the smallest exact group g*
from measure import CX_fig3_dominance
CX_fig3_dominance.main()

In [ ]:
###===================================================<<< Settings

system       = 'Common'

###===================================================<<< Calculation

# the circuit against the classical path sum, element by element
from measure import CX_exactness_direct
CX_exactness_direct.SYSTEM = system
CX_exactness_direct.main()